# NCDM-based Cognitive Skill Discovery

This notebook demonstrates the complete workflow used in the paper:

1. Load game metadata and user results
2. Filter users with at least 30 completed tasks
3. Apply z-score normalization and binarization
4. Train the NCDM model
5. Compare the discovered skill structure to the expert-defined Q-matrix
6. Evaluate stability across multiple trainings

In [1]:
import pandas as pd
import numpy as np
from utils import *

In [2]:
MIN_SOLVED_TASKS = 30

SUCCESS_THRESHOLD = 0

NUM_RUNS = 20

STABILITY_THRESHOLD = 0.3

In [124]:
df1 = pd.read_csv("example_Q.csv",  encoding="utf-8", sep=";")
df2 = pd.read_csv('example_dataset.csv', encoding="utf-8", sep = ",")

In [125]:
# items DataFrame: Q-matrix in a pandas DataFrame
items = pd.DataFrame({
    'item_id': range(1, len(df1) + 1),
    'knowledge_code': df1['AbilityAreaID'].values
})


## Single Training Run

First, we train the model and compare the discovered skills to the expert-defined Q-matrix.

In [ ]:
item2knowledge, knowledge_set = def_items(items)

train_data, val_data = split_train_valid(df2)

item_counts = train_data["item_id"].value_counts().sort_index()

num_items = train_data["item_id"].max() + 1

item_weights = torch.ones(num_items)

for item_id, freq in item_counts.items():
    item_weights[item_id] = 1.0 / np.sqrt(freq)   
item_weights /= item_weights.mean()

user_n, item_n, knowledge_n = num_user_item_knowledge(
        train_data, val_data, knowledge_set
    )

train, valid = transform_to_set(
        train_data, val_data, item2knowledge, knowledge_n
    )

model = NCDM_EarlyStopping.NCDM(knowledge_n, item_n, user_n)

train_model(model, train, valid, knowledge_n, item_n, user_n, item_weights)


Compare the predictions with the original Q-matrix

In [ ]:
compare_df = pd.DataFrame(columns = ["original", "prediction"])

for i in range (len(model.ncdm_net.k_difficulty.weight.detach().cpu().numpy())):

        row = model.ncdm_net.k_difficulty.weight[i].detach().numpy()

        # if a skill is considered relevant above a certain threshold

        #compare_df.loc[len(compare_df)] = [items.loc[i]['knowledge_code'], np.where(row > 0.3)[0]+1] 
        
        #top 3 skills
        k = 3
        topk = np.argsort(row)[-k:] + 1
        compare_df.loc[len(compare_df)] =[items.loc[i]['knowledge_code'], topk]


## Stability Analysis

To assess robustness, the complete training procedure is repeated 20 times.

In [ ]:
compare_df = pd.DataFrame(columns=["original"])
compare_df["original"] = items["knowledge_code"]

item2knowledge, knowledge_set = def_items(items)

train_data, val_data = split_train_valid(results)
item_counts = train_data["item_id"].value_counts().sort_index()
num_items = train_data["item_id"].max() + 1 
item_weights = torch.ones(num_items)
for item_id, freq in item_counts.items():
    item_weights[item_id] = 1.0 / np.sqrt(freq)
item_weights /= item_weights.mean()

all_probs = []  #additional information : probabilities/certanities of the predictions

for i in range(NUM_RUNS):
    
    actual, probs = training(items, results, item_weights)

    all_probs.append(probs)

    compare_df[f"soft_{i}"] = actual["soft"]

In [ ]:
all_probs = np.array(all_probs)

Stability

In [ ]:
freq_soft = compute_skill_frequency(compare_df)

stable_soft = extract_stable_skills(freq_soft, NUM_RUNS, min_ratio = STABILITY_THRESHOLD)

compare_df["stable_soft"] = stable_soft

In [ ]:
stable_compare_df_20 = compare_df[['original', 'stable_soft']] #final dataframe containing the original skills and the stable model-predicted skills after 20 trainings